In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

model = ChatOpenAI(model = 'gpt-4o-mini')

In [ ]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description= 'Detailed feedback for the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
essay = """Feminism is a social, political, and intellectual movement that advocates for the equality of women and men in all aspects of life. Throughout history, women have faced discrimination, limited rights, and unequal opportunities in education, employment, politics, and society. Feminism emerged as a response to these inequalities and has evolved through different phases, commonly referred to as "waves" of feminism. Each wave addressed specific issues and contributed significantly to improving the status of women worldwide.

The roots of feminism can be traced back to the late eighteenth century during the Age of Enlightenment. One of the earliest advocates for women's rights was Mary Wollstonecraft, whose book *A Vindication of the Rights of Woman* (1792) argued that women deserved the same educational opportunities as men. During this period, women began questioning traditional gender roles and demanding recognition as equal members of society. Although these early efforts did not immediately bring widespread change, they laid the foundation for future feminist movements.

The first wave of feminism emerged in the nineteenth and early twentieth centuries, primarily focusing on legal rights and political equality. Women campaigned for the right to vote, own property, receive education, and participate in public life. The suffrage movement became the central focus of this wave. Activists such as Susan B. Anthony, Elizabeth Cady Stanton, and Emmeline Pankhurst played crucial roles in advocating for women's voting rights. Their persistent efforts led to significant achievements, including the granting of voting rights to women in several countries. In the United States, women gained the right to vote in 1920 through the Nineteenth Amendment, while similar victories occurred in many other nations during the early twentieth century.

The second wave of feminism began in the 1960s and continued through the 1980s. While the first wave focused primarily on legal rights, the second wave addressed broader social, cultural, and economic inequalities. Feminists challenged workplace discrimination, unequal pay, reproductive rights restrictions, and traditional gender expectations. Influential works such as Betty Friedan's *The Feminine Mystique* inspired many women to question societal norms that confined them to domestic roles. During this period, women fought for equal employment opportunities, access to higher education, and greater control over their reproductive health. The second wave significantly expanded the scope of feminism and brought women's issues into mainstream public discussion.

The third wave of feminism emerged in the 1990s as a response to perceived limitations of earlier feminist movements. Third-wave feminists emphasized diversity, individuality, and intersectionality, recognizing that women's experiences differ based on race, ethnicity, class, sexuality, and other social factors. This wave challenged the notion that all women share the same experiences and highlighted the importance of inclusivity. Feminists worked to address issues affecting marginalized groups and promoted a more comprehensive understanding of gender equality.

In the twenty-first century, feminism entered what many scholars describe as the fourth wave. This phase is characterized by the widespread use of digital platforms and social media to raise awareness about gender-based discrimination, harassment, and violence. Campaigns such as #MeToo brought global attention to issues of sexual harassment and assault, empowering survivors to share their experiences and demand accountability. Modern feminism also advocates for equal pay, reproductive rights, political representation, LGBTQ+ rights, and the elimination of gender-based violence. The internet has enabled activists to connect across borders and mobilize support for feminist causes on a global scale.

Despite significant progress, challenges to gender equality remain. Women in many parts of the world continue to face discrimination, unequal access to education and healthcare, wage gaps, and underrepresentation in leadership positions. Feminism continues to evolve as it responds to changing social conditions and emerging issues. The movement seeks not only to improve the lives of women but also to create a more just and equitable society for everyone.

In conclusion, the history of feminism reflects a long and ongoing struggle for equality and human rights. From the early writings of Mary Wollstonecraft to modern digital activism, feminist movements have challenged discrimination and expanded opportunities for women around the world. Through its various waves, feminism has transformed societies, influenced laws, and promoted the principle that all individuals deserve equal rights and opportunities regardless of gender.
"""

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'

structured_model.invoke(prompt)


In [ ]:
class UPSEState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individial_scores: Annotated[list[int], operator.add]
    avg_score: float


In [ ]:
def evaluate_language(state: UPSEState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_analysis(state: UPSEState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def evaluate_thought(state: UPSEState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [ ]:
def final_evaluation(state: UPSEState):
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis feedback - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content

    average_score = sum(state["individial_scores"])/len(state["individial_scores"])

    return {'overall_feedback': overall_feedback, 'avg_score': average_score}

In [ ]:
graph = StateGraph(UPSEState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [ ]:
initial_state = {
    'essay': essay,
}

workflow.invoke(initial_state)
